# The HWP fast axis offset, from a polarized standard

A calibration run, not a science one. It reduces a sequence on a star whose
polarization angle on sky is already known, asks what `theta_off` makes the
pipeline reproduce that angle, and reports the answer. What you do with it is
hand it to the science reduction:

```toml
fast_axis_method = "fixed"
theta_off = <the number this prints>
```

which is why none of it is wired into `recipe.run`: the standard is a
different target from the science target, usually a different night, and the
offset is the only thing that travels between them.

The other route to the same number is
`nirc2pol.polarimetry.fit_fast_axis_butterfly`, which reads it off the
orientation of a tangentially polarized disk. Neither checks the other in the
sense of sharing assumptions — the butterfly needs a disk and assumes it is
azimuthally polarized, this needs a catalogue and assumes that catalogue —
which is exactly what makes agreement between them worth something.

This reports the **well-posed** fit: the offset with the instrumental
leakage left in. Solving for the leakage as well is possible and sometimes
better, but it needs the field to have rotated, and it fails in a way that
looks like an answer — so it is available here rather than automatic, and
section 5 says what it costs.

In [ ]:
import logging
import os

import numpy as np

from nirc2pol.polarimetry import (PolarizedStandard, curve_of_growth_polarization,
                                  fit_theta_off_polstd, measure_cycles,
                                  prepare_cycles)
from nirc2pol.recipe import run
from nirc2pol.reduction.config import ReductionConfig

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

# The config for the standard-star night.
CONFIG_PATH = "reduction_config.toml"

## 1. What is known about the star

From outside this pipeline. **There is no bundled catalogue on purpose** — a
value written here can be traced to whoever chose it, and one shipped in the
package cannot.

If the catalogue value is optical, carry it to the observing band with
`nirc2pol.polarimetry.serkowski_p` first, and put an honest `p_err` on the
result: at L′ that extrapolation is worth a factor rather than a percent.

`theta_err` is the one that matters. It is what the fit weights by, and **the
catalogue angle propagates into `theta_off` at half its own error** — 2° here
is 1° on the answer. `p` and `theta` are not interchangeable inputs: `p` sets
the modulus of the fitted coefficient and so lands in the efficiency, while
`theta` sets its phase and so lands in `theta_off`. A badly extrapolated `p`
leaves `theta_off` almost untouched.

In [ ]:
STANDARD = PolarizedStandard(
    name="REPLACE ME",
    p=0.0093,               # fraction, not percent, in the observed band
    theta=103.0,            # sky position angle [deg]
    p_err=0.004,
    theta_err=2.0,
    band="Lp",
    reference="fill this in -- it is the one input that cannot be recomputed",
)

# The aperture. Read the radius off the curve of growth below rather than
# choosing it here: p is a ratio over a shared aperture, so PSF clipping
# cannot bias it, and a p that moves with radius is telling you about the
# background instead.
RADIUS = 30.0
BACKGROUND = (110.0, 165.0)     # annulus for the residual background plane
MASK = None                     # exclude a companion or a registration wedge

## 2. Reduce the night

`run` does the standard reduction and hands back what it built. `prepare_cycles`
then takes the matched cycles to the point where `theta_off` is still free, so
trying a value is one rotation rather than a re-reduction.

`register_method` comes from the config so the photometry sees the same
registration the products did. Nothing else to choose: `measure_cycles` uses
only the instrument-to-sky frame rotation, never the north angle, so the
north-up question does not arise for an aperture sum.

In [ ]:
cfg = ReductionConfig.from_toml(CONFIG_PATH)
products = run(cfg, config_path=CONFIG_PATH)
instrument, cycles = products["instrument"], products["cycles"]

prepared = prepare_cycles(instrument, cycles,
                          register_method=cfg.register_method)
print(f"{len(prepared)} complete HWP cycles")

## 3. Is the measurement stable?

The standing check on the background treatment, and the reason to run it
before reading any fit.

**Both columns should be flat.** Aperture losses cannot bias `p` — `q`, `u`
and `I` are integrated over the same pixels, so clipping the PSF divides out.
So if `p` climbs with radius or the angle walks, that is not the source: it is
the background, and no fit below is worth reading until it is fixed.

The usual culprit is a residual in **Q and U**, not in I. They are
differences, so the sky is expected to cancel — and it does not, quite. On the
2025-12-06 standard, U carried a detector-scale gradient worth +22 ADU/px at
the star: nothing beside a 1.2e5 core, but an annulus at r=60–80 holds ~9000
pixels and sums it into more signal than the star has out there. Left in, `p`
ran from 0.9% to 4.1% across this table.

In [ ]:
cog = curve_of_growth_polarization(prepared, cfg.theta_off,
                                   radii=[10, 20, 30, 40, 60, 80],
                                   background=BACKGROUND, mask=MASK)
print("curve of growth in polarization (theta_off held at the config value)")
print(f"  {'r [px]':>7} {'p [%]':>9} {'theta [deg]':>12}")
for r, p, t in zip(cog["radius"], cog["p"], cog["theta"]):
    print(f"  {r:7.0f} {100 * p:9.3f} {t:12.2f}")

## 4. How noisy is one cycle?

This decides whether the joint IP fit can work at all. The offset and the
leakage separate through field rotation, and the separation is amplified by
the condition number — so a large per-cycle scatter at a short rotation span
means the joint fit returns noise however well posed it looks.

In [ ]:
measured = measure_cycles(prepared, radius=RADIUS, background=BACKGROUND,
                          mask=MASK)
sky = measured.z * np.exp(-1j * np.radians(measured.base
                                           + 4.0 * cfg.theta_off))
sigma = float(np.hypot(sky.real.std(), sky.imag.std()) / np.sqrt(2))

print(f"aperture r = {RADIUS:.0f} px at ({measured.center[0]:.2f}, "
      f"{measured.center[1]:.2f})")
print(f"  measured p = {100 * abs(sky.mean()):.3f}%, "
      f"sky angle = {np.degrees(0.5 * np.angle(sky.mean())) % 180:.2f} deg "
      f"at theta_off = {cfg.theta_off}")
print(f"  per-cycle scatter on q, u = {100 * sigma:.3f}%")

## 5. The fit

The model is one line, per HWP cycle `k`:

```
z_k = ip + c * s_k        s_k = p * exp(i(2*theta_known + base_k))
                          c   = efficiency * exp(i * 4*theta_off)
```

- `z_k` is what you measured — the aperture `q + iu` for that cycle, in the
  instrument frame.
- `s_k` is what the star should look like, and is **fully known**: `p` and
  `theta_known` from the catalogue, `base_k` from the telescope geometry.
- `c` is the unknown you want. Its modulus is the polarimetric efficiency,
  its phase is `4 * theta_off`.
- `ip` is the instrumental I → Q/U leakage, constant in the *instrument*
  frame.

Linear in `ip` and `c`, so it is a closed-form least squares — no optimizer.

### The two options

**`fit_ip=False`** (used below) sets `ip = 0` and solves for `c` alone. One
complex unknown from N complex measurements, so it is always well
conditioned. Any leakage that is actually present does not vanish: it is
absorbed into `c`, and displaces `theta_off`.

**`fit_ip=True`** solves for `ip` and `c` together. It removes that
displacement, and multiplies the noise by the condition number of the fit.

What separates the two terms is that **`s_k` rotates with the field and `ip`
does not** — that asymmetry is the entire handle. With little field rotation
the two design columns are nearly parallel, and the split between them is
close to arbitrary.

### Reading the diagnostics

- **efficiency** is the fraction of the true polarization recovered, so it
  lies in (0, 1].
- **condition number** measures how nearly parallel those two columns are.
- **rotation span** is how far the field turned across the sequence.

`fit_theta_off_polstd`'s docstring carries a synthetic table of the bias and
scatter of each option against span and per-cycle noise.

In [ ]:
fit = fit_theta_off_polstd(prepared, STANDARD, radius=RADIUS,
                           background=BACKGROUND, mask=MASK)

print(f"theta_off   {fit.theta_off:+8.3f} +/- {fit.theta_off_err:.3f} deg")
print(f"efficiency  {fit.efficiency:8.3f}")
print(f"rotation    {fit.rotation_span:8.1f} deg over {len(prepared)} cycles")

# To solve for the leakage as well, uncomment. Compare theta_off, the
# efficiency and the condition number against the fit above before using it.
# joint = fit_theta_off_polstd(prepared, STANDARD, radius=RADIUS,
#                              background=BACKGROUND, mask=MASK, fit_ip=True)
# print(f"\njoint: theta_off {joint.theta_off:+.3f} "
#       f"+/- {joint.theta_off_err:.3f} deg, "
#       f"efficiency {joint.efficiency:.3f}, "
#       f"ipq {joint.ip.ipq:+.4f}, ipu {joint.ip.ipu:+.4f}, "
#       f"condition number {joint.condition_number:.0f}")

## 6. Report the dependence, not just the number

The catalogue angle is the input least likely to be right, and the whole
dependence on it is one line. Print the line rather than only the point
estimate, and the next person can re-evaluate it when a better value turns up
without reducing the night again.

In [ ]:
reference_angle = float(np.degrees(0.5 * np.angle(sky.mean())) % 180.0)
print("For any other catalogue angle, without re-reducing:")
print(f"    theta_off = {cfg.theta_off} - (theta_known - "
      f"{reference_angle:.2f}) / 2")